In [1]:
import pandas as pd
from tabulate import tabulate
import os
import statsmodels.api as sm
import numbers
import matplotlib.pyplot as plt
from io import StringIO
import warnings
from statsmodels.tools.sm_exceptions import ValueWarning

warnings.filterwarnings("ignore", category=ValueWarning)
path = "img/Semana 9"

def print_tabulate(df:pd.DataFrame)-> pd.DataFrame:
    print(tabulate(df, headers= df.columns, tablefmt='orgtbl'))

df = pd.read_csv("csv/aprovCreditos.csv", index_col=False)
os.makedirs(path, exist_ok=True)

colNumericas = df.select_dtypes(include='number').columns.to_list()

colCategoricas = ["Program", "Country", "Term"]


def transform_variable(df: pd.DataFrame, x: str) -> pd.Series:
    if isinstance(df[x][0], numbers.Number):
        return df[x]
    else:
        return pd.Series([i for i in range(0, len(df[x]))])

def linear_regression(df: pd.DataFrame, x: str, y: str) -> None:
    fixed_x = transform_variable(df, x)
    model = sm.OLS(df[y], sm.add_constant(fixed_x)).fit()
    print(f"R2: {model.rsquared_adj}")

    coef = pd.read_html(StringIO(model.summary().tables[1].as_html()), header=0, index_col=0)[0]['coef']
    df.plot(x=x, y=y, kind='scatter')
    plt.plot(df[x], [coef.values[1] * x + coef.values[0] for _, x in fixed_x.items()], color='red')
    plt.xticks(rotation=90)
    plt.savefig(f'{path}/lr_{y}_{x}.png')
    plt.close()


print(f'Graficas generadas en: {path}')
for cn in colNumericas:
    for col in colCategoricas:
        if col == cn:
            continue
        print(f'{col} y {cn}')
        df_agg = df.groupby([col])[[cn]].agg({cn: ['sum']})
        df_agg.reset_index(inplace=True)

        df_agg.columns = [col,f"{cn} suma"]
        df_agg.reset_index(inplace=True)

        linear_regression(df_agg, col, f"{cn} suma")


Graficas generadas en: img/Semana 9
Program y Approved_Declined_Amount
R2: 0.49604848136313295
Country y Approved_Declined_Amount
R2: 0.00797998820878787
Term y Approved_Declined_Amount
R2: -0.23255451077893063
Program y Disbursed_Shipped_Amount
R2: 0.8301742213770644
Country y Disbursed_Shipped_Amount
R2: 0.0039324886743472565
Term y Disbursed_Shipped_Amount
R2: -0.6865240838685656
Program y Decision_Day
R2: -0.43548124229762064
Country y Decision_Day
R2: -0.0037542066841524946
Term y Decision_Day
R2: 0.5417141037983884
Program y Decision_Month
R2: -0.43379708754174295
Country y Decision_Month
R2: -0.003757462753251284
Term y Decision_Month
R2: 0.5493919658591535
Program y Decision_Year
R2: -0.43193353386409505
Country y Decision_Year
R2: -0.0038876759788604165
Term y Decision_Year
R2: 0.5452466463790082
